# Coordinator — Signal Table (Notebook 01)

Build the aligned historical signal table: all 4 agents × all 4 pairs × 2022–2025.
One cell at a time — verify each step before proceeding.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Root:", ROOT)
print("Exists:", ROOT.exists())

Root: D:\SCRIPTS\FX-AlphaLab
Exists: True


## 1. Load Technical Agents

In [2]:
from src.agents.technical.agent import TechnicalAgent, fuse_timeframe_signals

PAIRS = ["EURUSDm", "GBPUSDm", "USDJPYm", "USDCHFm"]
TIMEFRAMES = ["D1", "H4", "H1"]

tech_agents = {}
for pair in PAIRS:
    tech_agents[pair] = {}
    for tf in TIMEFRAMES:
        path = ROOT / "models" / "technical" / f"{pair}_{tf}.pkl"
        tech_agents[pair][tf] = TechnicalAgent.load(path)

print("Loaded technical agents:")
for pair in PAIRS:
    print(f"  {pair}: {list(tech_agents[pair].keys())}")

Loaded technical agents:
  EURUSDm: ['D1', 'H4', 'H1']
  GBPUSDm: ['D1', 'H4', 'H1']
  USDJPYm: ['D1', 'H4', 'H1']
  USDCHFm: ['D1', 'H4', 'H1']


In [3]:
import pandas as pd

# Sanity check: one predict for one pair
test_date = pd.Timestamp("2024-06-15", tz="UTC")
test_pair = "EURUSDm"

signals = {tf: tech_agents[test_pair][tf].predict(test_pair, test_date) for tf in TIMEFRAMES}
fused = fuse_timeframe_signals(signals)

print(f"Fused signal for {test_pair} on {test_date.date()}:")
print(f"  direction={fused.direction}  confidence={fused.confidence:.4f}")
print(f"  volatility_regime={fused.volatility_regime}")
print(f"  timeframe_votes={fused.timeframe_votes}")

Fused signal for EURUSDm on 2024-06-15:
  direction=0  confidence=0.0169
  volatility_regime=low
  timeframe_votes={'D1': 0, 'H4': 1, 'H1': 0}


## 2. Load Geopolitical Agent

In [4]:
from src.agents.geopolitical.agent import GeopoliticalAgent

geo_agent = GeopoliticalAgent()
geo_agent.load()
print("Geopolitical agent loaded.")

2026-05-02 19:54:00,355 - GeopoliticalAgent - INFO - Loaded zone features: 3454 rows


Geopolitical agent loaded.


In [5]:
import pandas as pd

geo_sig = geo_agent.predict("EURUSD", pd.Timestamp("2024-06-15", tz="UTC"))
print(f"bilateral_risk_score={geo_sig.bilateral_risk_score:.4f}  risk_regime={geo_sig.risk_regime}")

bilateral_risk_score=1.6838  risk_regime=high


## 3. Load Macro Agent

In [8]:
from src.agents.macro.agent import MacroAgent
from src.agents.macro.calendar_node import CalendarEventsNode

calendar_node = CalendarEventsNode()
macro_agent = MacroAgent(calendar_node=calendar_node)
macro_agent.load()
print("Macro agent loaded.")

2026-05-02 20:14:16,331 - MacroeconomicsNode - INFO - Loaded macro signal data: 224 rows from D:\SCRIPTS\FX-AlphaLab\data\processed\macro\macro_signal.parquet


2026-05-02 20:14:16,331 - MacroeconomicsNode - INFO - Loaded macro signal data: 224 rows from D:\SCRIPTS\FX-AlphaLab\data\processed\macro\macro_signal.parquet


2026-05-02 20:14:16,358 - CalendarEventsNode - INFO - Calendar event scores loaded from cache (32683 rows).


2026-05-02 20:14:16,358 - CalendarEventsNode - INFO - Calendar event scores loaded from cache (32683 rows).


2026-05-02 20:14:16,362 - CalendarEventsNode - INFO - Calendar events scored for EURUSD: 6661


...

2026-05-02 20:14:16,365 - CalendarEventsNode - INFO - Calendar events scored for USDCHF: 7453


2026-05-02 20:14:16,365 - CalendarEventsNode - INFO - Calendar events scored for USDCHF: 7453


2026-05-02 20:14:16,367 - CalendarEventsNode - INFO - Calendar events scored for USDJPY: 8712


2026-05-02 20:14:16,367 - CalendarEventsNode - INFO - Calendar events scored for USDJPY: 8712


Macro agent loaded.


In [9]:
macro_sig = macro_agent.predict("EURUSDm", pd.Timestamp("2024-06-15", tz="UTC"))
print(f"direction={macro_sig.module_c_direction}  confidence={macro_sig.macro_confidence:.4f}")
print(f"surprise_score={macro_sig.macro_surprise_score:.4f}  dominant_driver={macro_sig.dominant_driver}")

direction=down  confidence=1.0000
surprise_score=0.0000  dominant_driver=regime_context_score
